In [2]:
import torch
import torch.nn as nn # importing the neural network module from PyTorch
import torch.nn.functional as F # importing the functional module from PyTorch for activation functions and other
import torch.optim as optim # importing the optimization module from PyTorch for training the model
from torch.utils.data import DataLoader, TensorDataset # importing DataLoader and TensorDataset for handling data in batches

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm # importing tqdm for displaying progress bars during training

In [3]:
device = "cuda" if torch.cuda.is_available() else "cpu" # setting the device to GPU if available, otherwise CPU
print(f"Using device: {device}") # printing the device being used

Using device: cuda


In [4]:
X, y = load_breast_cancer(return_X_y=True) # loading the breast cancer dataset and splitting it into features (X) and labels (y)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42) # splitting the dataset into training and testing sets
scaler = StandardScaler() # creating an instance of the StandardScaler for feature scaling

In [5]:
X_train_scaled = scaler.fit_transform(X_train) # fitting the scaler to the training data and transforming it
X_test_scaled = scaler.transform(X_test) # transforming the test data using the fitted scaler


In [6]:
X_train_scaled 

array([[-1.44075296, -0.43531947, -1.36208497, ...,  0.9320124 ,
         2.09724217,  1.88645014],
       [ 1.97409619,  1.73302577,  2.09167167, ...,  2.6989469 ,
         1.89116053,  2.49783848],
       [-1.39998202, -1.24962228, -1.34520926, ..., -0.97023893,
         0.59760192,  0.0578942 ],
       ...,
       [ 0.04880192, -0.55500086, -0.06512547, ..., -1.23903365,
        -0.70863864, -1.27145475],
       [-0.03896885,  0.10207345, -0.03137406, ...,  1.05001236,
         0.43432185,  1.21336207],
       [-0.54860557,  0.31327591, -0.60350155, ..., -0.61102866,
        -0.3345212 , -0.84628745]], shape=(455, 30))

In [7]:
y

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0,
       0, 0, 1, 0, 1, 1, 1, 1, 1, 0, 0, 1, 0, 0, 1, 1, 1, 1, 0, 1, 0, 0,
       1, 1, 1, 1, 0, 1, 0, 0, 1, 0, 1, 0, 0, 1, 1, 1, 0, 0, 1, 0, 0, 0,
       1, 1, 1, 0, 1, 1, 0, 0, 1, 1, 1, 0, 0, 1, 1, 1, 1, 0, 1, 1, 0, 1,
       1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 1, 0, 0, 1, 1, 1, 0, 0, 1, 0, 1, 0,
       0, 1, 0, 0, 1, 1, 0, 1, 1, 0, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 0, 1, 1, 1, 1, 0, 0, 1, 0, 1, 1, 0, 0, 1, 1, 0, 0, 1, 1, 1,
       1, 0, 1, 1, 0, 0, 0, 1, 0, 1, 0, 1, 1, 1, 0, 1, 1, 0, 0, 1, 0, 0,
       0, 0, 1, 0, 0, 0, 1, 0, 1, 0, 1, 1, 0, 1, 0, 0, 0, 0, 1, 1, 0, 0,
       1, 1, 1, 0, 1, 1, 1, 1, 1, 0, 0, 1, 1, 0, 1, 1, 0, 0, 1, 0, 1, 1,
       1, 1, 0, 1, 1, 1, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 1, 1, 1, 1, 1, 1, 0, 1, 0, 1, 1, 0, 1, 1, 0, 1, 0, 0, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 0,

In [8]:
X_train_scaled_tensor = torch.tensor(X_train_scaled, dtype=torch.float32, device=device) # converting the scaled training features to a PyTorch tensor
x_test_scaled_tensor = torch.tensor(X_test_scaled, dtype=torch.float32, device=device) # converting the scaled test features to a PyTorch tensor
y_train_tensor = torch.tensor(y_train, dtype=torch.float32, device=device).unsqueeze(1) # converting the training labels to a PyTorch tensor and adding an extra dimension for compatibility with the model
y_test_tensor = torch.tensor(y_test, dtype=torch.float32, device=device).unsqueeze(1) # converting the test labels to a PyTorch tensor and adding an extra dimension for compatibility with the model  

# squeeze for removing dimensions of size 1, unsqueeze for adding dimensions of size 1, and view for reshaping tensors. 
#unsqueeze(index postion)

In [9]:
train_dataset = TensorDataset(X_train_scaled_tensor, y_train_tensor) # creating a TensorDataset for the training data
test_dataset = TensorDataset(x_test_scaled_tensor, y_test_tensor) # creating a TensorDataset

X_train_scaled_tensor.shape 

torch.Size([455, 30])

In [10]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True) # creating a DataLoader for the training dataset with a batch size of 32 and shuffling enabled
test_loader = DataLoader(test_dataset, batch_size=32) # creating a DataLoader for

In [11]:
class BreastCancerModel(nn.Module): # defining a neural network model class that inherits from nn.Module
    def __init__(self): # initializing the model
        super().__init__() # calling the constructor of the parent class: keep parent class setup(logic, attributes, etc.)
        self.fc1 = nn.Linear(30, 16) # Liner(input, output): defining the first fully connected layer with 30 input features and 16 output features
        self.fc2 = nn.Linear(16, 8) # defining the second fully connected layer with 16 input features and 8 output features
        self.fc3 = nn.Linear(8, 1) # defining the third fully connected layer with 8 input features and 1 output feature (for binary classification)
        self.dropout = nn.Dropout(0.2) # defining a dropout layer with a dropout probability of 0.2 to prevent overfitting
        
    def forward(self, x): # defining the forward pass of the model : forward mainly for feeding the data through the layers
        x = F.relu(self.fc1(x)) # applying ReLU activation function after the first layer
        x = F.relu(self.fc2(x)) # applying ReLU activation function after the second layer
        x = self.dropout(x) # applying dropout to the output of the second layer
        x = torch.sigmoid(self.fc3(x)) # applying sigmoid activation function after the third layer to get output between 0 and 1
        return x

In [12]:
model = BreastCancerModel().to(device) # creating an instance of the BreastCancerModel and moving it to the specified device (GPU or CPU)

In [13]:
criterion = nn.BCELoss() # defining the loss function as Binary Cross Entropy Loss for binary classification
optimizer = optim.Adam(model.parameters(), lr=0.001) # defining the optimizer as Adam with a learning rate of 0.001 to update the model parameters during training

In [14]:
epochs = 100 # setting the number of training epochs to 100
for epoch in range(epochs): # loop over the number of epochs
    model.train() # setting the model to training mode
    for X_batch, y_batch in train_loader: # iterating over the training data in batches
        optimizer.zero_grad() # clearing the gradients of the optimizer
        outputs = model(X_batch) # getting the model's predictions for the current batch
        loss = criterion(outputs, y_batch) # calculating the loss between the predictions and the true labels
        loss.backward() # performing backpropagation to compute gradients
        optimizer.step() # updating the model parameters based on the computed gradients
    print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

Epoch 1, Loss: 0.7127
Epoch 2, Loss: 0.6902
Epoch 3, Loss: 0.6058
Epoch 4, Loss: 0.5606
Epoch 5, Loss: 0.5292
Epoch 6, Loss: 0.4413
Epoch 7, Loss: 0.4614
Epoch 8, Loss: 0.4191
Epoch 9, Loss: 0.4742
Epoch 10, Loss: 0.1152
Epoch 11, Loss: 0.4998
Epoch 12, Loss: 0.1569
Epoch 13, Loss: 0.0436
Epoch 14, Loss: 0.6031
Epoch 15, Loss: 0.1356
Epoch 16, Loss: 0.1625
Epoch 17, Loss: 0.0784
Epoch 18, Loss: 0.1317
Epoch 19, Loss: 0.1261
Epoch 20, Loss: 0.2266
Epoch 21, Loss: 0.0610
Epoch 22, Loss: 0.0059
Epoch 23, Loss: 0.0691
Epoch 24, Loss: 0.1591
Epoch 25, Loss: 0.0112
Epoch 26, Loss: 0.0677
Epoch 27, Loss: 0.0202
Epoch 28, Loss: 0.0084
Epoch 29, Loss: 0.0286
Epoch 30, Loss: 0.0194
Epoch 31, Loss: 0.1161
Epoch 32, Loss: 0.0093
Epoch 33, Loss: 0.0819
Epoch 34, Loss: 0.1303
Epoch 35, Loss: 0.0226
Epoch 36, Loss: 0.0004
Epoch 37, Loss: 0.1152
Epoch 38, Loss: 0.0807
Epoch 39, Loss: 0.0171
Epoch 40, Loss: 0.0179
Epoch 41, Loss: 0.0343
Epoch 42, Loss: 0.0020
Epoch 43, Loss: 0.0170
Epoch 44, Loss: 0.00

In [20]:
with torch.no_grad(): # disabling gradient calculation for evaluation
    model.eval() # setting the model to evaluation 
    preds = model(x_test_scaled_tensor) # getting the model's predictions for the test data
    predicted = (preds >= 0.5)
    accuracy = (predicted == y_test_tensor).float().mean()
    print(f"Test Accuracy: {accuracy.item():.4f}")

Test Accuracy: 0.9825


In [23]:
y_test_tensor

tensor([[1.],
        [0.],
        [0.],
        [1.],
        [1.],
        [0.],
        [0.],
        [0.],
        [1.],
        [1.],
        [1.],
        [0.],
        [1.],
        [0.],
        [1.],
        [0.],
        [1.],
        [1.],
        [1.],
        [0.],
        [0.],
        [1.],
        [0.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [0.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [0.],
        [1.],
        [0.],
        [1.],
        [1.],
        [0.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [0.],
        [0.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [0.],
        [0.],
        [1.],
        [1.],
        [0.],
        [0.],
        [1.],
        [1.],
        [1.],
        [0.],
        [0.],
        [1.],
        [1.],
        [0.],
      